# Maximizar la Luminosidad de un Disco de Acreción Simplificado

## Descripción del problema

Se considera un disco de acreción alrededor de un objeto compacto cuya emisión
depende del radio interno $r$ y de un parámetro angular $\theta$ asociado a la
inclinación efectiva del flujo. La luminosidad neta está dada por:

$$L = f(r, \theta) = r^2 \sin\theta\,(1 + \cos\theta)\,e^{-r/r_0}$$

con $r_0 = 1$ por simplicidad.

## Restricciones físicas

| Variable | Dominio | Interpretación |
|----------|---------|----------------|
| $r$ | $(0,\, 5r_0]$ (numérico: $[0.1,\, 5]$) | Radio físico del disco |
| $\theta$ | $[0,\, \pi/2]$ | Ángulos físicamente relevantes |

## Condición inicial — Caso c)

$$f(r_0,\,\theta_0) = (0.5\,r_0,\; \pi/3)$$

es decir, $r_0 = 0.5$ y $\theta_0 = \pi/3 \approx 1.047$ rad.

In [9]:
import numpy as np
import matplotlib.pyplot as plt
r0 = 1
def L(r, theta):
    return r**2 * np.sin(theta) * (1 + np.cos(theta)) * np.exp(-r / r0)

## Metodo de Optimizacion directa multidimensiona

In [17]:


# Parámetros




def ciclo_coordenadas(f, r0_init, theta0_init, tol=1e-6, max_iter=1000):
    """Optimización por ciclo de coordenadas (coordinate descent) para maximizar f."""
    r, theta = r0_init, theta0_init

    def golden_max_1d(g, a, b, eps=1e-8):
        phi = (np.sqrt(5) - 1) / 2
        x1 = b - phi * (b - a)
        x2 = a + phi * (b - a)
        f1, f2 = g(x1), g(x2)
        while (b - a) > eps:
            if f1 < f2:
                a = x1; x1 = x2; f1 = f2
                x2 = a + phi * (b - a); f2 = g(x2)
            else:
                b = x2; x2 = x1; f2 = f1
                x1 = b - phi * (b - a); f1 = g(x1)
        return (a + b) / 2

    historial = [(r, theta, f(r, theta))]

    for i in range(max_iter):
        r_prev, theta_prev = r, theta

        # Optimizar sobre r fijando theta
        r = golden_max_1d(lambda rv: f(rv, theta), 0.1, 5 * r0)

        # Optimizar sobre theta fijando r
        theta = golden_max_1d(lambda tv: f(r, tv), 0, np.pi / 2)

        historial.append((r, theta, f(r, theta)))

        if abs(r - r_prev) < tol and abs(theta - theta_prev) < tol:
            print(f"Convergió en {i+1} iteraciones")
            break

    return r, theta, f(r, theta), historial

# Caso c): punto inicial (0.5*r0, π/3)
r_init    = 0.5 * r0
theta_init = np.pi / 3

r_opt, theta_opt, L_max, hist = ciclo_coordenadas(L, r_init, theta_init)

print(f"Punto inicial : r={r_init:.4f}, θ={theta_init:.4f} rad")
print(f"r óptimo      : {r_opt:.6f}")
print(f"θ óptimo      : {theta_opt:.6f} rad  ({np.degrees(theta_opt):.4f}°)")
print(f"L máxima      : {L_max:.6f}")



Convergió en 2 iteraciones
Punto inicial : r=0.5000, θ=1.0472 rad
r óptimo      : 2.000000
θ óptimo      : 1.047198 rad  (60.0000°)
L máxima      : 0.703223


# Metodo del gradiente "Descenso mas pronunciado"

In [15]:

from scipy.optimize import minimize_scalar


def grad_L(r, theta):
    e = np.exp(-r / r0)
    s, c = np.sin(theta), np.cos(theta)
    dL_dr     = (2*r - r**2 / r0) * s * (1 + c) * e
    dL_dtheta = r**2 * e * (c*(1+c) - s**2)
    return np.array([dL_dr, dL_dtheta])

def steepest_ascent(r_init, theta_init, tol=1e-7, max_iter=1000):
    x = np.array([r_init, theta_init], dtype=float)

    for i in range(max_iter):
        g = grad_L(x[0], x[1])

        if np.linalg.norm(g) < tol:
            break

        # Line search: encontrar h* que maximiza L(x + h·g)
        def phi(h):
            x_new = x + h * g
            x_new[0] = np.clip(x_new[0], 0.1, 5 * r0)
            x_new[1] = np.clip(x_new[1], 0.0, np.pi / 2)
            return -L(x_new[0], x_new[1])   # negativo porque minimize_scalar minimiza

        res  = minimize_scalar(phi, bounds=(0, 1), method='bounded')
        h_opt = res.x

        x_new    = x + h_opt * g
        x_new[0] = np.clip(x_new[0], 0.1, 5 * r0)
        x_new[1] = np.clip(x_new[1], 0.0, np.pi / 2)
        x        = x_new

    return x[0], x[1], L(x[0], x[1]), i+1

# Caso c): punto inicial (0.5·r0, π/3)
r_opt, theta_opt, L_max, iters = steepest_ascent(0.5 * r0, np.pi / 3)

print(f"Iteraciones : {iters}")
print(f"r óptimo    : {r_opt:.6f}")
print(f"θ óptimo    : {theta_opt:.6f} rad  ({np.degrees(theta_opt):.4f}°)")
print(f"L máxima    : {L_max:.6f}")

Iteraciones : 36
r óptimo    : 2.000000
θ óptimo    : 1.047198 rad  (60.0000°)
L máxima    : 0.703223
